# Carbon melt-quench example
*Copyright (c) 2026 Miguel A. Caro*

This is a helper script for the simple melt-quench simulation example of the **TurboGAP School**.
We will embed the TurboGAP simulations into this notebook (they are usually run directly in the terminal)
and will visualize what's happening to the structures.

Have a terminal open side by side with this notebook in case you need to edit a **TurboGAP** input file.

You're not supposed to senselessly run all the cells below in sequence! That might even throw
out some errors! Instead, go step by step and try to understand what's going on. This notebook will
also ask you to run some tasks in the Terminal before continuing with the notebook code.

READ ALL THE COMMENTS IN THE CELLS BEFORE RUNNING THE CODE, THEY MAY BE ASKING YOU TO DO SOMETHING!!!

In [ ]:
# Module imports
from ase.io import read # Used to generate an ASE Atoms object from various XYZ files
from weas_widget import WeasWidget # The visualization widget to take a look at the structures
import numpy as np # Numpy is always useful
import matplotlib as mpl # For plots
import matplotlib.pyplot as plt # For plots

## Melt
1. We start by generating the initial structure by running `python make_initial.py` in the terminal. You can also
copy the contents of that script here and run it, but this is perhaps a cleaner way. This will
generate a file called `atoms0.xyz` with an initial structure. If you want to change the initial structure parameters,
open and edit `make_initial.py` with the terminal.

In [ ]:
! python make_initial.py

2. Let's visualize this initial structure

In [ ]:
# We define a helper function for viewing the atoms
def view(atoms):
    viewer = WeasWidget()
    viewer.avr.model_style = 1 # ball & stick mode
    viewer.from_ase(atoms)
    viewer.avr.show_bonded_atoms = True # show bonds across periodic boundaries
    return viewer

atoms = read("atoms0.xyz")
view(atoms)

3. Now, we copy the **TurboGAP** input file for a melt simulation from the `input_files` directory,
`cp input_files/melt input` and run a **TurboGAP** MD simulation, `mpirun -np 3 turbogap md`. On Noppe, the
virtual machines *seem* to have 14 CPUs but you can't use that many cores as they are shared!
Here, we are using 3 CPU cores per user, by passing the number after the `-np` flag, to
get acceleration from parallelization. This simulation should take less than 1 min if you left the 30 atoms
in the orginal `make_initial.py` script.

In [ ]:
! cp input_files/melt input
! mpirun -np 3 turbogap md

4. Let's visualize the trajectory, which in **TurboGAP** is always stored as `trajectory_out.xyz`.

In [ ]:
atoms = read("trajectory_out.xyz", index=":") # run with index="-1" so see the last snapshot only
view(atoms)

5. Let us now visualize some of the logged information of the simulation, i.e., temperature, pressure, energy.
This info is collected in the "thermo.log" file. A basic thermo.log file has 6 columns: 1) step number, 2) time (fs), 3) temperature (K), 4) kinetic energy (eV), 5) potential energy (eV), 6) pressure (bar). Let us load the datafile:

In [ ]:
log = np.loadtxt("thermo.log")
#    and do some plotting
fig, axs = plt.subplots(2, 3, figsize=(10,5))
axs[0,0].plot(log[:,1], log[:,0]); axs[0,1].plot(log[:,1], log[:,1]); axs[0,2].plot(log[:,1], log[:,2])
axs[1,0].plot(log[:,1], log[:,3]); axs[1,1].plot(log[:,1], log[:,4]); axs[1,2].plot(log[:,1], log[:,5])
for i in range(0,2):
    for j in range(0,3):
        axs[i,j].set_xlabel("Time (fs)")
axs[0,0].set_ylabel("Time step"); axs[0,1].set_ylabel("Time (fs)"); axs[0,2].set_ylabel("Temperature (K)")
axs[1,0].set_ylabel("Kinetic energy (eV)"); axs[1,1].set_ylabel("Potential energy (eV)"); axs[1,2].set_ylabel("Pressure (bar)")
fig.tight_layout()
plt.show()

## Liquid
1. We will now generate liquid carbon. For that let us save time by restarting from the last snapshot from the previous
simulation. You can do this here with ASE if you know how to. You can also do it in the terminal, for instance with
the bash command `tail`. This is the preferred method if you are running a high-throughput workflow and don't want
to waste time. First, check how many atoms *N* your structure has, then use `tail` to extract the last *N+2* lines from the
`trajectory_out.xyz` file from the previous simulation. E.g, if you have 30 atoms, do `tail -32 trajectory_out.xyz > atoms.xyz`.

NOTE: we are going to overwrite **TurboGAP** input and output files. If you want to save some of the files, you need to do
that now before proceeding with the next step.

In [ ]:
! tail -32 trajectory_out.xyz > atoms.xyz

2. Now copy the liquid input file `cp input_files/liquid input` and run `mpirun -np 3 turbogap md`.

In [ ]:
! cp input_files/liquid input
! mpirun -np 3 turbogap md

3. Let's take a look at the new trajectory:

In [ ]:
atoms = read("trajectory_out.xyz", index=":") # run with index="-1" so see the last snapshot only
view(atoms)

4. You can observe that the box is now also changing its size because we are using a barostat. We enabled logging of lattice vector
information (columns 7 to 15 in the log file), so let's take a look at that:

In [ ]:
log = np.loadtxt("thermo.log")
#    and do some plotting
fig, axs = plt.subplots(2, 3, figsize=(10,5))
axs[0,0].plot(log[:,1], log[:,2]); axs[0,1].plot(log[:,1], log[:,4]); axs[0,2].plot(log[:,1], log[:,5])
axs[1,0].plot(log[:,1], log[:,6]); axs[1,1].plot(log[:,1], log[:,10]); axs[1,2].plot(log[:,1], log[:,14])
for i in range(0,2):
    for j in range(0,3):
        axs[i,j].set_xlabel("Time (fs)")
axs[0,0].set_ylabel("Temperature (K)"); axs[0,1].set_ylabel("Potential energy (eV)"); axs[0,2].set_ylabel("Pressure (bar)")
axs[1,0].set_ylabel("a (Angstrom)"); axs[1,1].set_ylabel("b (Angstrom)"); axs[1,2].set_ylabel("c (Angstrom)")
fig.tight_layout()
plt.show()

You may or may not know this, but our numbers are very far from being sensible due to the very short MD runs; we've
kept them short so that we can proceed reasonably fast within this hands-on session. E.g., the time coupling constants
for the thermostat and barostat are too small by about 1 and 2 orders of magnitude, respectively, compared to usual
settings. Next, we will take (a bit) more time and make something more reasonable.

## Melt-quench simulation
1. In a melt-quench simulation, we play with the various MD steps, temperature profiles, and pressure profiles to
access different parts of the vast space of metastable carbon configurations. Here, I encourage you to think about
what kind of carbon material you may want to generate and what kind of simulation parameters may induce that. Note
that, because we are using an MD-based approach, the materials will need to arise spontaneously from the simulation.
This means that low-entropy structures, like fullerene, might be too difficult to achieve. However, low-entropy
structures that are also very low in free energy (like diamond at high pressure) might still be feasible. Think
about: 1) temperature profile, 2) pressure profile, 3) elastic anisotropicity (for the barostat), 4) number of
atoms needed, e.g., to capture disorder, 5) target density, 6) sp2 (graphitic) vs sp3 (diamondlike) structure, etc.
You can even try and make better liquid carbon than we made above. You can also think about technical details like
the MD step: if you can make it longer, you can run longer MD for the same cost; if it's too long, your system
might blow up! Likewise, if you can lower the number of atoms in the simulation that will make them faster (but
will prevent you from accessing truly disordered structures. Always remember that **TurboGAP** will not automatically
start from the last snapshot of a previous calculation: you need to extract this snapshop and provide it as the new
initial structure. You can also do a few parallel-efficiency tests to check with how many cores you can run fast.
2. The teacher has prepared a two-step optimization of a graphitic carbon for himself, and will start from the last
snapshot of the liquid carbon simulation. These are `input_files/teacher1` and `input_files/teacher2`. You can use
them for inspiration but try to make your own, as we will take a look at all the structures at the end and it would
be nice to have a variety.

In [ ]:
! # This is what the teacher is running. Modify the input files and run with your own simulation parameters!
! # Melt step
! tail -32 trajectory_out.xyz > atoms.xyz # get last snapshot from the liquid carbon simulation
! cp input_files/teacher1 input
! mpirun -np 3 turbogap md

In [ ]:
atoms = read("trajectory_out.xyz", index=":") # run with index="-1" so see the last snapshot only
view(atoms)
log = np.loadtxt("thermo.log")
#    and do some plotting
fig, axs = plt.subplots(2, 3, figsize=(10,5))
axs[0,0].plot(log[:,1], log[:,2]); axs[0,1].plot(log[:,1], log[:,4]); axs[0,2].plot(log[:,1], log[:,5])
axs[1,0].plot(log[:,1], log[:,6]); axs[1,1].plot(log[:,1], log[:,10]); axs[1,2].plot(log[:,1], log[:,14])
for i in range(0,2):
    for j in range(0,3):
        axs[i,j].set_xlabel("Time (fs)")
axs[0,0].set_ylabel("Temperature (K)"); axs[0,1].set_ylabel("Potential energy (eV)"); axs[0,2].set_ylabel("Pressure (bar)")
axs[1,0].set_ylabel("a (Angstrom)"); axs[1,1].set_ylabel("b (Angstrom)"); axs[1,2].set_ylabel("c (Angstrom)")
fig.tight_layout()
plt.show()

In [ ]:
! # Quench step
! tail -32 trajectory_out.xyz > atoms.xyz # get last snapshot from the melt run
! cp input_files/teacher2 input
! mpirun -np 3 turbogap md

In [ ]:
atoms = read("trajectory_out.xyz", index=":") # run with index="-1" so see the last snapshot only
view(atoms)
log = np.loadtxt("thermo.log")
#    and do some plotting
fig, axs = plt.subplots(2, 3, figsize=(10,5))
axs[0,0].plot(log[:,1], log[:,2]); axs[0,1].plot(log[:,1], log[:,4]); axs[0,2].plot(log[:,1], log[:,5])
axs[1,0].plot(log[:,1], log[:,6]); axs[1,1].plot(log[:,1], log[:,10]); axs[1,2].plot(log[:,1], log[:,14])
for i in range(0,2):
    for j in range(0,3):
        axs[i,j].set_xlabel("Time (fs)")
axs[0,0].set_ylabel("Temperature (K)"); axs[0,1].set_ylabel("Potential energy (eV)"); axs[0,2].set_ylabel("Pressure (bar)")
axs[1,0].set_ylabel("a (Angstrom)"); axs[1,1].set_ylabel("b (Angstrom)"); axs[1,2].set_ylabel("c (Angstrom)")
fig.tight_layout()
plt.show()

3. Once you have your structure, you can use the command `submit_structure` so that the teacher can gather them also and
run some analysis.

In [ ]:
! tail -32 trajectory_out.xyz > atoms.xyz # get last snapshot
! submit_structure

## Analyze the structures from all the students

1. With the command `retrieve_structures`, a directory will be generated and filled with the structures the students submitted.

In [ ]:
! retrieve_structures

2. Once that is done, we can run the following code to get some statistics:

In [ ]:
import os
filelist = os.listdir("student_structures")
from ase.neighborlist import NeighborList

data = []
for file in filelist:
    atoms = read("student_structures/%s" % (file))
    name = file[0:-4]
    nl = NeighborList(1.85/2.+np.zeros(len(atoms)), skin=0., sorted=True, self_interaction=False, bothways=True)
    nl.update(atoms)
    n_sp2 = 0; n_sp3 = 0
    for i in range(0, len(atoms)):
        idx = nl.get_neighbors(i)[0]
        if len(idx) == 3:
            n_sp2 += 1
        elif len(idx) == 4:
            n_sp3 += 1
    T = atoms.info["temperature"]
    P = atoms.info["pressure"]
    V = atoms.info["volume"]; rho = (len(atoms) * 12.01 * 1.660539) / V
    E = atoms.get_potential_energy()
    data.append([name, rho, n_sp2/len(atoms), n_sp3/len(atoms), P, T, E/len(atoms)])

def annotate(ax, ix, iy, data):
    for dat in data:
        ax.annotate(" " + dat[0], xy = (dat[ix], dat[iy]), color="blue", fontsize=8)

fig, axs = plt.subplots(2, 3, figsize=(10,5))
axs[0,0].plot([dat[2] for dat in data], [dat[3] for dat in data], marker="x", color="red", linestyle="None"); axs[0,0].set_xlabel("sp2 fraction"); axs[0,0].set_ylabel("sp3 fraction")
annotate(axs[0,0], 2, 3, data)
axs[0,1].plot([dat[2] for dat in data], [dat[1] for dat in data], marker="x", color="red", linestyle="None"); axs[0,1].set_xlabel("sp2 fraction"); axs[0,1].set_ylabel("Density (g/cm^3)")
annotate(axs[0,1], 2, 1, data)
axs[0,2].plot([dat[3] for dat in data], [dat[1] for dat in data], marker="x", color="red", linestyle="None"); axs[0,2].set_xlabel("sp3 fraction"); axs[0,2].set_ylabel("Density (g/cm^3)")
annotate(axs[0,2], 3, 1, data)
axs[1,0].plot([dat[4] for dat in data], [dat[1] for dat in data], marker="x", color="red", linestyle="None"); axs[1,0].set_xlabel("Pressure (bar)"); axs[1,0].set_ylabel("Density (g/cm^3)")
annotate(axs[1,0], 4, 1, data)
axs[1,1].plot([dat[4] for dat in data], [dat[3] for dat in data], marker="x", color="red", linestyle="None"); axs[1,1].set_xlabel("Pressure (bar)"); axs[1,1].set_ylabel("sp3 fraction")
annotate(axs[1,1], 4, 3, data)
axs[1,2].plot([dat[6] for dat in data], [dat[2] for dat in data], marker="x", color="red", linestyle="None"); axs[1,2].set_xlabel("Energy per atom (eV)"); axs[1,2].set_ylabel("sp2 fraction")
annotate(axs[1,2], 6, 2, data)

fig.tight_layout()
plt.show()